In [1]:
import numpy as np
import xarray as xr
from glob import glob
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
dsList = []

for year in range(2014, 2025):
    print(year)
    folder = f'/proj/cmip6/data/ocean_reanalysis/glorys12v1/{year:04d}/'
    ds = xr.open_mfdataset(glob(folder + '*.nc'))
    
    sst = ds.thetao.isel(depth=0)
    sst = sst.sel(latitude = 0, method='nearest')
    
    uo = ds.uo.isel(depth=0)
    uo = uo.sel(latitude = 0, method='nearest')
    
    zos = ds.zos.sel(latitude = 0, method='nearest')
    
    mlotst = ds.mlotst.sel(latitude =0, method='nearest')
    
    ds.close()
    
    subds = xr.Dataset()
    subds['uo'] = uo
    subds['sst'] = sst
    subds['zos'] = zos
    subds['mld'] = mlotst
    
    dsList.append(subds)
    

2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [3]:
fullDS = xr.concat(dsList, dim='time').compute()

In [4]:
fullDS

<xarray.Dataset>
Dimensions:    (longitude: 4320, time: 4011)
Coordinates:
  * longitude  (longitude) float32 -180.0 -179.9 -179.8 ... 179.8 179.8 179.9
    latitude   float32 0.0
    depth      float32 0.494
  * time       (time) datetime64[ns] 2014-01-01T12:00:00 ... 2024-12-24
Data variables:
    uo         (time, longitude) float32 -0.7648 -0.744 ... -0.4218 -0.4199
    sst        (time, longitude) float32 27.91 27.9 27.89 ... 27.31 27.3 27.29
    zos        (time, longitude) float32 0.6091 0.6091 0.6088 ... 0.5673 0.5664
    mld        (time, longitude) float32 24.57 24.72 24.57 ... 35.55 35.55 35.4

In [5]:
writeFname = '../../WPWP_GLORYS_data/equator_sst_u_ssh_mld_2014_2024.nc'
new_lon = (fullDS['longitude'].values + 360) % 360
fullDS = fullDS.assign_coords(longitude=new_lon)
fullDS = fullDS.sortby('longitude')
fullDS = fullDS.sel(longitude=slice(106, 296))

# Write immediately
fullDS.to_netcdf(writeFname, unlimited_dims='time')

In [6]:
fullDS

<xarray.Dataset>
Dimensions:    (longitude: 2281, time: 4011)
Coordinates:
  * longitude  (longitude) float32 106.0 106.1 106.2 106.2 ... 295.8 295.9 296.0
    latitude   float32 0.0
    depth      float32 0.494
  * time       (time) datetime64[ns] 2014-01-01T12:00:00 ... 2024-12-24
Data variables:
    uo         (time, longitude) float32 0.1068 0.09766 0.09888 ... nan nan nan
    sst        (time, longitude) float32 27.59 27.58 27.59 27.58 ... nan nan nan
    zos        (time, longitude) float32 0.9955 0.9952 0.9949 ... nan nan nan
    mld        (time, longitude) float32 35.86 36.01 37.84 36.32 ... nan nan nan